# Phase 2.4: LSTM Training - 24h Horizon (96 timesteps)

## Objectif
Entraîner LSTM pour prédiction long terme (24h, 96 timesteps).

## Architecture
- Input: (batch, 60, n_pollutants)
- LSTM 64 + Dropout 0.2 + return_sequences=True
- LSTM 32 + Dropout 0.2
- Dense 16 + ReLU
- Dense 96 (output)

## Sortie
- `ia/models/model_lstm_24h.h5`: Modèle Keras
- `ia/models/lstm_24h_metrics.json`: Métriques (RMSE, MAE, MAPE, R²)

## Section 1: Setup et Chargement Tenseurs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import pickle

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
tf.random.set_seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"✅ Imports réussis")
print(f"   TensorFlow version: {tf.__version__}")

In [ ]:
# Configuration des chemins
PROJECT_ROOT = Path("../").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

tensors_file = DATA_DIR / "lstm_train_val_test.pkl"
model_file = MODELS_DIR / "model_lstm_24h.h5"
metrics_file = MODELS_DIR / "lstm_24h_metrics.json"
history_file = MODELS_DIR / "lstm_24h_history.json"

print(f"📦 Tensors input: {tensors_file}")
print(f"💾 Model output: {model_file}")
print(f"📊 Metrics output: {metrics_file}")

In [ ]:
# Charger tenseurs préparés
print("🔄 Chargement tenseurs...\n")

with open(tensors_file, 'rb') as f:
    lstm_data = pickle.load(f)

# Extraire tenseurs pour horizon 24h (96 timesteps)
HORIZON = 96
combined = lstm_data['combined_tensors'][HORIZON]

X_train = combined['train'][0]
y_train = combined['train'][1]
X_val = combined['val'][0]
y_val = combined['val'][1]
X_test = combined['test'][0]
y_test = combined['test'][1]

print(f"✅ Tenseurs chargés pour horizon {HORIZON} (24h)")
print(f"\n   Dimensions:")
print(f"   X_train: {X_train.shape} (batch, lookback={X_train.shape[1]}, pollutants={X_train.shape[2]})")
print(f"   y_train: {y_train.shape} (batch, horizon={y_train.shape[1]})")
print(f"   X_val: {X_val.shape}")
print(f"   y_val: {y_val.shape}")
print(f"   X_test: {X_test.shape}")
print(f"   y_test: {y_test.shape}")

## Section 2: Architecture LSTM

In [ ]:
# ========================================
# SECTION 2: Architecture LSTM (24h)
# ========================================
# Même architecture que LSTM 1h, mais avec Dense(96) en sortie
# pour prédire 96 timesteps (≈ 24 heures) au lieu de 4

print("🏗️  Construction architecture LSTM (24h)...\\n")

# Architecture:
# Input → LSTM(64) → Dropout → LSTM(32) → Dropout → Dense(16) → Dense(96) → Output
#
# DIFFÉRENCE avec LSTM 1h:
# - Dense(96) au lieu de Dense(4)
# - 96 points = 96 heures ≈ 24 heures (si données horaires)
# - Tâche plus difficile: prédire plus loin dans le temps

model = keras.Sequential([
    # Couche d'entrée: (batch, lookback=60, n_pollutants=8)
    layers.Input(shape=(X_train.shape[1], X_train.shape[2])),
    
    # Première couche LSTM: 64 unités
    layers.LSTM(64, return_sequences=True, activation='relu'),
    layers.Dropout(0.2),
    
    # Deuxième couche LSTM: 32 unités
    layers.LSTM(32, activation='relu'),
    layers.Dropout(0.2),
    
    # Couches denses
    layers.Dense(16, activation='relu'),
    layers.Dense(96)  # HORIZON=96 pour 24h prediction
])

# Compiler avec mêmes hyperparamètres que LSTM 1h
model.compile(
    loss='mse',
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=['mae']
)

print(f"✅ Modèle LSTM 24h construit")
print(f"   Architecture: LSTM(64) → LSTM(32) → Dense(16) → Dense(96)")
print(f"   Paramètres: {model.count_params():,}")
print(f"\nSummaire du modèle:")
model.summary()

## Section 3: Entraînement

In [ ]:
# Entraîner modèle avec early stopping
print("🔄 Entraînement LSTM (horizon 24h)...\n")

early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

print(f"\n✅ Entraînement terminé")

In [ ]:
# Visualiser courbes d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_title('LSTM 24h - Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].grid(alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Val MAE', linewidth=2)
axes[1].set_title('LSTM 24h - MAE', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'lstm_24h_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Courbes d'entraînement visualisées")

## Section 4: Évaluation

In [ ]:
# ========================================
# SECTION 4: Évaluation sur Test Set
# ========================================
# Mesurer la performance du modèle sur données jamais vues (test set)

print("🔍 Évaluation sur test set...\\n")

# Prédire les valeurs de test
y_test_pred = model.predict(X_test)

# CALCUL DES MÉTRIQUES:
# 
# 1. MSE (Mean Squared Error) = moyenne((réel - prédit)²)
#    - Penalise les gros erreurs davantage
#
# 2. RMSE (Root Mean Squared Error) = √MSE
#    - Même unités que les données d'origine
#    - Cible: RMSE < 10% de la moyenne
#
# 3. MAE (Mean Absolute Error) = moyenne(|réel - prédit|)
#    - Erreur moyenne absolue
#    - Plus interprétable que MSE
#
# 4. MAPE (Mean Absolute Percentage Error) = moyenne(|réel - prédit| / |réel|)
#    - Pourcentage d'erreur (permet comparaison entre datasets)
#    - Cible: MAPE < 10%
#
# 5. R² (Coefficient of Determination)
#    - 1.0 = parfait, 0.0 = juste la moyenne, <0 = pire que moyenne
#    - Cible: R² > 0.85

mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_test_pred)
mape = np.mean(np.abs((y_test - y_test_pred) / y_test)) * 100
r2 = r2_score(y_test, y_test_pred)

print(f"✅ Métriques sur test set:")
print(f"   RMSE: {rmse:.4f}")
print(f"   MAE:  {mae:.4f}")
print(f"   MAPE: {mape:.2f}%")
print(f"   R²:   {r2:.4f}")

# Valider contre les targets de la spec
print(f"\n   Validation vs targets spec:")
print(f"   RMSE < 10%: {'✅' if rmse < 0.1 else '❌'}")
print(f"   R² > 0.85: {'✅' if r2 > 0.85 else '⚠️'}  (important: peut être <0.85 pour long-term)")
print(f"   MAPE < 10%: {'✅' if mape < 10 else '⚠️'}")

In [ ]:
# Visualiser prédictions vs réel (premiers 100 samples, toute la séquence 96pts)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 4 exemples de prédictions complets
for idx, ax in enumerate(axes.flatten()):
    sample_idx = idx * (len(y_test) // 4)
    if sample_idx < len(y_test):
        steps = np.arange(HORIZON)
        ax.plot(steps, y_test[sample_idx], marker='o', label='Real', linewidth=2, markersize=4)
        ax.plot(steps, y_test_pred[sample_idx], marker='s', label='Prediction', linewidth=2, markersize=4, alpha=0.7)
        ax.set_title(f'Sample {sample_idx}: 24h Prediction', fontsize=11, fontweight='bold')
        ax.set_xlabel('Hour')
        ax.set_ylabel('Normalized Value')
        ax.legend()
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'lstm_24h_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Visualisation prédictions sauvegardée")

In [ ]:
# Distribution des erreurs de prédiction
errors = np.abs(y_test - y_test_pred).flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(errors, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(np.mean(errors), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(errors):.4f}')
axes[0].axvline(np.median(errors), color='orange', linestyle='--', linewidth=2, label=f'Median: {np.median(errors):.4f}')
axes[0].set_title('Distribution des Erreurs Absolues', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Erreur Absolue')
axes[0].set_ylabel('Fréquence')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Q-Q plot (quantile-quantile)
from scipy import stats
stats.probplot(errors, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot (Normalité des Erreurs)', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'lstm_24h_error_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Distribution erreurs visualisée")

## Section 5: Sauvegarde

In [ ]:
# Sauvegarder modèle
print("💾 Sauvegarde du modèle...\n")

model.save(model_file)
print(f"✅ Modèle sauvegardé: {model_file}")

# Sauvegarder metrics
metrics = {
    'horizon': 96,
    'horizon_hours': 24,
    'rmse': float(rmse),
    'mae': float(mae),
    'mape': float(mape),
    'r2_score': float(r2),
    'test_samples': len(y_test),
    'epochs_trained': len(history.history['loss']),
    'error_mean': float(np.mean(errors)),
    'error_median': float(np.median(errors)),
    'error_std': float(np.std(errors))
}

with open(metrics_file, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"✅ Métriques sauvegardées: {metrics_file}")

# Sauvegarder history
history_dict = {
    'loss': history.history['loss'],
    'val_loss': history.history['val_loss'],
    'mae': history.history['mae'],
    'val_mae': history.history['val_mae']
}

with open(history_file, 'w') as f:
    json.dump(history_dict, f, indent=2)

print(f"✅ History sauvegardé: {history_file}")

## Section 6: Comparaison avec LSTM 1h

In [ ]:
# Charger métriques du modèle 1h pour comparaison
print("📊 Comparaison LSTM 1h vs 24h...\n")

metrics_1h_file = MODELS_DIR / "lstm_1h_metrics.json"

if metrics_1h_file.exists():
    with open(metrics_1h_file, 'r') as f:
        metrics_1h = json.load(f)
    
    comparison = pd.DataFrame([
        {
            'Model': 'LSTM 1h',
            'RMSE': metrics_1h['rmse'],
            'MAE': metrics_1h['mae'],
            'MAPE (%)': metrics_1h['mape'],
            'R²': metrics_1h['r2_score']
        },
        {
            'Model': 'LSTM 24h',
            'RMSE': metrics['rmse'],
            'MAE': metrics['mae'],
            'MAPE (%)': metrics['mape'],
            'R²': metrics['r2_score']
        }
    ])
    
    print("Comparaison Modèles:")
    print(comparison.to_string(index=False))
else:
    print("⚠️  Modèle 1h non trouvé, comparaison skip")

## ✅ Résumé - LSTM 24h (Horizon 96)

✅ Modèle LSTM construit: LSTM(64) → LSTM(32) → Dense(16) → Dense(96)
✅ Entraîné sur 70% des données (lookback=60)
✅ Validé sur 15%, testé sur 15% (split temporel strict)
✅ Early stopping appliqué (patience=10)
✅ Métriques finales:
   - RMSE: {rmse:.4f}
   - MAE: {mae:.4f}
   - MAPE: {mape:.2f}%
   - R²: {r2:.4f}
✅ Modèle sauvegardé (.h5 format Keras)
✅ Comparaison 1h vs 24h effectuée

## 🎯 Prochaines Étapes
1. Notebook 08: Inference testing (charger modèles + valider latence)
2. Créer `ia/api.py` (FastAPI service)
3. Créer `backend/AIService.js` (Node.js integration)
4. Notebook 09: Validation E2E (tests end-to-end)